## Part IX — Agent Demo 

### Overview

The idea here is that the AI Agent can be provided a general, plain text description of an individual insurance applicant and eventually output a predicted premium and predicted claim value. However, unlike in "04. LLM," this code instructs the LLM to utilize the trained h1 and h2 models as needed to output a more accurate result. The LLM serves only as a wrapper to call specialized tools.

RF is preferred for premium calculation (regression) and MLR is preferred for claim likelihood (classification), as we determined in our earlier testing.

### Step 1: Load Saved Artifacts

In [1]:
# ── Step 1: Imports & Load Artifacts ────────────────────────────────────────
import joblib
import numpy as np
import pandas as pd
import ollama
from IPython.display import Markdown, display

# ── PCA / clustering artifacts (Notebook 1) ──────────────────────────────────
scaler   = joblib.load('scaler.pkl')    # StandardScaler for PCA inputs only
pca      = joblib.load('pca.pkl')       # 2-component PCA on vehicle specs
kmeans   = joblib.load('kmeans.pkl')    # KMeans, 4 clusters
tier_map = joblib.load('tier_map.pkl')  # {2: 'Budget', 1: 'Standard', 0: 'Premium', 3: 'High Performance'}

# ── MLR artifacts (Notebook 2) ───────────────────────────────────────────────
mlr_premium     = joblib.load('mlr_premium.pkl')      # OLS → log(Premium)
mlr_claim       = joblib.load('lr_claim_balanced.pkl')  # balanced classifier (imbalance-corrected)
scaler_mlr      = joblib.load('scaler_mlr.pkl')        # StandardScaler for MLR continuous cols
continuous_cols = joblib.load('continuous_cols.pkl')   # which cols scaler_mlr applies to

# ── RF artifacts (Notebook 3) ────────────────────────────────────────────────
rf_premium = joblib.load('rf_premium.pkl')   # RF → log(Premium)

print("All artifacts loaded.")
print(f"  tier_map:        {tier_map}")
print(f"  continuous_cols: {continuous_cols}")

All artifacts loaded.
  tier_map:        {2: 'Budget', 1: 'Standard', 0: 'Premium', 3: 'High Performance'}
  continuous_cols: ['Seniority', 'Policies_in_force', 'Max_policies', 'Max_products', 'N_claims_history', 'R_Claims_history', 'N_doors', 'Driver_age', 'Licence_years', 'Vehicle_age']


### Step 2 — Core Feature Vector Builder

In [2]:
# ── Step 2: Feature Vector Builder ──────────────────────────────────────────
# This function replicates the full preprocessing pipeline for a single new policy.
# It is the bridge between human-readable inputs and the model-ready feature matrix.
#
# PCA inputs: Power, Cylinder_capacity, Weight, Length, Value_vehicle
#   → scaled with scaler.pkl → projected with pca.pkl → cluster with kmeans.pkl
#   → Risk_tier string via tier_map.pkl
#   → one-hot encoded as Risk_tier_Standard / Risk_tier_Premium / Risk_tier_High Performance
#   (Budget is the dropped reference category from drop_first=True)

PCA_COLS = ['Power', 'Cylinder_capacity', 'Weight', 'Length', 'Value_vehicle']

def build_feature_vector(
    # Driver features
    driver_age:          int,
    licence_years:       int,
    seniority:           int,
    n_claims_history:    int,
    r_claims_history:    float,
    is_active:           int,   # 1 = active policy, 0 = lapsed
    # Policy/account features
    policies_in_force:   int,
    max_policies:        int,
    max_products:        int,
    second_driver:       int,   # 1 = yes
    area:                int,   # 0 or 1
    type_risk:           int,   # 1, 2, 3, or 4
    distribution_channel: int,  # 0 or 1
    payment:             int,   # 0 or 1
    # Vehicle features (fed into PCA → tier assignment)
    power:               float,  # kW
    cylinder_capacity:   float,  # cc
    weight:              float,  # kg
    length:              float,  # metres (as stored after preprocessing)
    value_vehicle:       float,  # €
    vehicle_age:         int,
    n_doors:             int,
    fuel_diesel:         int,   # 1 = diesel, 0 = petrol
) -> tuple:
    """
    Returns (X_raw, risk_tier):
        X_raw     — 22-column unscaled DataFrame (RF-ready)
        risk_tier — human-readable tier string ('Budget', 'Standard', 'Premium', 'High Performance')
    """
    # ── Assign risk tier via PCA → KMeans ───────────────────────────────────
    pca_input  = np.array([[power, cylinder_capacity, weight, length, value_vehicle]])
    pca_scaled = scaler.transform(pca_input)        # scaler.pkl (Notebook 1)
    pca_coords = pca.transform(pca_scaled)           # pca.pkl
    cluster    = kmeans.predict(pca_coords)[0]       # kmeans.pkl
    risk_tier  = tier_map[cluster]                   # tier_map.pkl

    # ── Build one-hot risk tier columns ────────────────────────────────────
    # Budget is drop_first reference — all zeros means Budget
    risk_hp       = 1 if risk_tier == 'High Performance' else 0
    risk_premium  = 1 if risk_tier == 'Premium'          else 0
    risk_standard = 1 if risk_tier == 'Standard'         else 0

    # ── Assemble the 22-column feature vector ───────────────────────────────
    row = {
        'Seniority':                seniority,
        'Policies_in_force':        policies_in_force,
        'Max_policies':             max_policies,
        'Max_products':             max_products,
        'N_claims_history':         n_claims_history,
        'R_Claims_history':         r_claims_history,
        'Second_driver':            second_driver,
        'N_doors':                  n_doors,
        'Is_active':                is_active,
        'Driver_age':               driver_age,
        'Licence_years':            licence_years,
        'Vehicle_age':              vehicle_age,
        'Fuel_diesel':              fuel_diesel,
        'Risk_tier_High Performance': risk_hp,
        'Risk_tier_Premium':         risk_premium,
        'Risk_tier_Standard':        risk_standard,
        'Area_1':                   1 if area == 1 else 0,
        'Type_risk_2':              1 if type_risk == 2 else 0,
        'Type_risk_3':              1 if type_risk == 3 else 0,
        'Type_risk_4':              1 if type_risk == 4 else 0,
        'Distribution_channel_1':   1 if distribution_channel == 1 else 0,
        'Payment_1':                1 if payment == 1 else 0,
    }

    X_raw = pd.DataFrame([row])
    return X_raw, risk_tier


print("build_feature_vector() defined.")

build_feature_vector() defined.


This creates a function that standardizes data pre-processing for model usage.

### Step 3 — Tool Functions

In [3]:
# ── Step 3: Tool Functions ────────────────────────────────────────────────────
# Task assignment based on model evaluation results:
#   predict_premium           → h1 (Random Forest): best regression performance
#   predict_claim_probability → h2 (Logistic Regression): AUC 0.91, best classification
#
# Default values on every parameter act as a safety net if Ollama fails to extract
# a specific field from the natural language prompt. The defaults represent a
# typical mid-risk policy and will not normally be used.

def predict_premium(
    driver_age: int            = 40,
    licence_years: int         = 20,
    seniority: int             = 3,
    n_claims_history: int      = 0,
    r_claims_history: float    = 0.0,
    is_active: int             = 1,
    policies_in_force: int     = 1,
    max_policies: int          = 1,
    max_products: int          = 1,
    second_driver: int         = 0,
    area: int                  = 0,
    type_risk: int             = 1,
    distribution_channel: int  = 0,
    payment: int               = 0,
    power: float               = 90.0,
    cylinder_capacity: float   = 1400.0,
    weight: float              = 1200.0,
    length: float              = 4.2,
    value_vehicle: float       = 12000.0,
    vehicle_age: int           = 5,
    n_doors: int               = 4,
    fuel_diesel: int           = 0
) -> dict:
    """
    Predicts annual insurance premium in euros using h1 (Random Forest).
    h1 outperformed h2 on regression. No feature scaling required —
    RF is invariant to feature magnitude.
    """
    X_raw, risk_tier = build_feature_vector(
        driver_age=driver_age, licence_years=licence_years, seniority=seniority,
        n_claims_history=n_claims_history, r_claims_history=r_claims_history,
        is_active=is_active, policies_in_force=policies_in_force,
        max_policies=max_policies, max_products=max_products,
        second_driver=second_driver, area=area, type_risk=type_risk,
        distribution_channel=distribution_channel, payment=payment,
        power=power, cylinder_capacity=cylinder_capacity, weight=weight,
        length=length, value_vehicle=value_vehicle,
        vehicle_age=vehicle_age, n_doors=n_doors, fuel_diesel=fuel_diesel
    )

    h2_premium = float(np.expm1(rf_premium.predict(X_raw)[0]))

    return {
        'model_used':              'h1 — Random Forest',
        'predicted_premium_euros': round(h2_premium, 2),
        'risk_tier':               risk_tier,
    }


def predict_claim_probability(
    driver_age: int            = 40,
    licence_years: int         = 20,
    seniority: int             = 3,
    n_claims_history: int      = 0,
    r_claims_history: float    = 0.0,
    is_active: int             = 1,
    policies_in_force: int     = 1,
    max_policies: int          = 1,
    max_products: int          = 1,
    second_driver: int         = 0,
    area: int                  = 0,
    type_risk: int             = 1,
    distribution_channel: int  = 0,
    payment: int               = 0,
    power: float               = 90.0,
    cylinder_capacity: float   = 1400.0,
    weight: float              = 1200.0,
    length: float              = 4.2,
    value_vehicle: float       = 12000.0,
    vehicle_age: int           = 5,
    n_doors: int               = 4,
    fuel_diesel: int           = 0
) -> dict:
    """
    Predicts claim probability using h2 (Logistic Regression).
    h2 outperformed h1 on classification (AUC 0.91).
    Continuous features are scaled with scaler_mlr before prediction.
    """
    X_raw, risk_tier = build_feature_vector(
        driver_age=driver_age, licence_years=licence_years, seniority=seniority,
        n_claims_history=n_claims_history, r_claims_history=r_claims_history,
        is_active=is_active, policies_in_force=policies_in_force,
        max_policies=max_policies, max_products=max_products,
        second_driver=second_driver, area=area, type_risk=type_risk,
        distribution_channel=distribution_channel, payment=payment,
        power=power, cylinder_capacity=cylinder_capacity, weight=weight,
        length=length, value_vehicle=value_vehicle,
        vehicle_age=vehicle_age, n_doors=n_doors, fuel_diesel=fuel_diesel
    )

    X_mlr = X_raw.copy()
    X_mlr[continuous_cols] = scaler_mlr.transform(X_raw[continuous_cols])
    h1_claim = float(mlr_claim.predict_proba(X_mlr)[0][1])

    return {
        'model_used':        'h2 — Logistic Regression',
        'claim_probability': round(h1_claim, 4),
        'risk_tier':         risk_tier,
    }


print("Tool functions defined.")
print("  predict_premium()           → h1 (Random Forest)")
print("  predict_claim_probability() → h2 (Logistic Regression)")

Tool functions defined.
  predict_premium()           → h1 (Random Forest)
  predict_claim_probability() → h2 (Logistic Regression)


Two tool functions are created that the LLM will be able to call-- one to predict claim probability and the other to predict the premium in Euros.

### Step 4 — Tool Schemas and Ollama Tool-Calling Loop

In [4]:
# ── Step 4: Tool Schemas & Ollama Tool-Calling Loop ──────────────────────────

POLICY_PARAMS_SCHEMA = {
    "type": "object",
    "properties": {
        "driver_age":            {"type": "integer", "description": "Driver age in years"},
        "licence_years":         {"type": "integer", "description": "Years the driver has held a licence"},
        "seniority":             {"type": "integer", "description": "Years as a customer with this insurer"},
        "n_claims_history":      {"type": "integer", "description": "Total number of claims in driver history"},
        "r_claims_history":      {"type": "number",  "description": "Historical claim rate (claims per year)"},
        "is_active":             {"type": "integer", "description": "1 if policy is currently active, 0 if lapsed"},
        "policies_in_force":     {"type": "integer", "description": "Number of policies currently active"},
        "max_policies":          {"type": "integer", "description": "Maximum number of policies ever held simultaneously"},
        "max_products":          {"type": "integer", "description": "Maximum number of product types ever held"},
        "second_driver":         {"type": "integer", "description": "1 if a second driver is listed, 0 otherwise"},
        "area":                  {"type": "integer", "description": "Geographic area code: 0 or 1"},
        "type_risk":             {"type": "integer", "description": "Risk category: 1, 2, 3, or 4"},
        "distribution_channel":  {"type": "integer", "description": "Sales channel: 0 or 1"},
        "payment":               {"type": "integer", "description": "Payment method: 0 or 1"},
        "power":                 {"type": "number",  "description": "Engine power in kilowatts (kW)"},
        "cylinder_capacity":     {"type": "number",  "description": "Engine displacement in cubic centimetres (cc)"},
        "weight":                {"type": "number",  "description": "Vehicle weight in kilograms (kg)"},
        "length":                {"type": "number",  "description": "Vehicle length in metres"},
        "value_vehicle":         {"type": "number",  "description": "Current market value of the vehicle in euros"},
        "vehicle_age":           {"type": "integer", "description": "Age of the vehicle in years"},
        "n_doors":               {"type": "integer", "description": "Number of doors (0 = 2-door, 4 = 4-door)"},
        "fuel_diesel":           {"type": "integer", "description": "1 if diesel fuel, 0 if petrol"},
    },
    "required": [
        "driver_age", "licence_years", "seniority",
        "n_claims_history", "r_claims_history", "is_active",
        "policies_in_force", "max_policies", "max_products",
        "second_driver", "area", "type_risk", "distribution_channel", "payment",
        "power", "cylinder_capacity", "weight", "length", "value_vehicle",
        "vehicle_age", "n_doors", "fuel_diesel"
    ]
}

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "predict_premium",
            "description": (
                "Predicts the annual insurance premium in euros using h1 (Random Forest), "
                "which was the best-performing model for regression. "
                "Call this when a premium estimate is needed."
            ),
            "parameters": POLICY_PARAMS_SCHEMA,
        }
    },
    {
        "type": "function",
        "function": {
            "name": "predict_claim_probability",
            "description": (
                "Predicts the probability that a claim will occur using h2 (Logistic Regression), "
                "which was the best-performing model for classification (AUC 0.91). "
                "Call this when a claim risk estimate is needed."
            ),
            "parameters": POLICY_PARAMS_SCHEMA,
        }
    },
]

TOOL_REGISTRY = {
    "predict_premium":           predict_premium,
    "predict_claim_probability": predict_claim_probability,
}

# ── Type coercion ─────────────────────────────────────────────────────────────
# Ollama returns all tool call arguments as strings regardless of schema types.
# This map casts them back to the correct Python types before calling the function.
SCHEMA_TYPES = {
    k: v['type']
    for k, v in POLICY_PARAMS_SCHEMA['properties'].items()
}

def coerce_args(raw_args: dict) -> dict:
    coerced = {}
    for k, v in raw_args.items():
        t = SCHEMA_TYPES.get(k)
        try:
            if t == 'integer':
                coerced[k] = int(float(v))   # float() first handles '1.0' edge case
            elif t == 'number':
                coerced[k] = float(v)
            else:
                coerced[k] = v
        except (ValueError, TypeError):
            coerced[k] = v                    # leave as-is if cast fails
    return coerced


def run_agent(policy_description: str, verbose: bool = True) -> str:
    """
    Runs the full tool-calling loop with Ollama (Llama 3.2).

    1. Sends the policy description + tool schemas to the LLM
    2. LLM returns tool call(s) with extracted arguments
    3. Python coerces argument types and executes each tool
    4. Results are appended to conversation history and returned to the LLM
    5. LLM generates the final underwriting memo

    Args:
        policy_description: natural language description of the policy to underwrite
        verbose: if True, prints intermediate tool calls and results

    Returns:
        Final memo text from the LLM
    """
    SYSTEM_PROMPT = (
        "You are an experienced motor insurance underwriter. "
        "When given a policy description, call predict_premium to get the premium estimate "
        "(Random Forest — our best regression model) and call predict_claim_probability "
        "to get the claim risk estimate (Logistic Regression — our best classification model). "
        "IMPORTANT: Only report numbers that were explicitly returned by the tools. "
        "Do not invent, estimate, or calculate any values yourself. "
        "Write a concise 5-7 sentence underwriting memo covering: the predicted premium, "
        "the predicted claim probability, what the risk tier implies, "
        "why each model was chosen for its task, and your underwriting recommendation. "
        "Always call both tools before writing the memo."
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": policy_description},
    ]

    # ── Round 1: LLM decides which tools to call ─────────────────────────────
    response = ollama.chat(
        model="llama3.2",
        messages=messages,
        tools=TOOLS,
    )

    # ── Tool execution loop ───────────────────────────────────────────────────
    while response.message.tool_calls:
        messages.append(response.message)

        for tool_call in response.message.tool_calls:
            fn_name = tool_call.function.name
            fn_args = coerce_args(tool_call.function.arguments)  # cast string → correct types

            if verbose:
                print(f"\n  → LLM called: {fn_name}")
                print(f"    args: {fn_args}")

            if fn_name in TOOL_REGISTRY:
                result = TOOL_REGISTRY[fn_name](**fn_args)
            else:
                result = {"error": f"Unknown tool: {fn_name}"}

            if verbose:
                print(f"    result: {result}")

            messages.append({
                "role":    "tool",
                "content": str(result),
            })

        # ── Subsequent rounds: LLM responds given tool results ────────────────
        response = ollama.chat(
            model="llama3.2",
            messages=messages,
            tools=TOOLS,
        )

    return response.message.content


print("run_agent() defined. Ollama must be running: `ollama serve`")

run_agent() defined. Ollama must be running: `ollama serve`


This tells Ollama what tools it can call, what the tool does, and what the tool needs in terms of data and structure. Ollama's 3.2 model inherently supports function / tool calling, but if that was not the case we could use an LLM orchestration library like LangChain.

### Step 5: Define Scenario Policies and Run

In [5]:
# ── Step 5: Define Scenario Policies ─────────────────────────────────────────
# Written as natural language prompts — the LLM parses these and extracts
# the structured arguments it needs to call the tools.

LOW_RISK_PROMPT = """
Please underwrite the following policy:

- Driver: 58 years old, has held a licence for 38 years
- Customer seniority: 7 years with this insurer
- Prior claims: 0 total, historical claim rate 0.0
- Policy is currently active
- Policies in force: 1, max policies ever held: 2, max products: 1
- No second driver listed
- Area: 0, risk type: 1, distribution channel: 0, payment method: 0
- Vehicle: petrol, 75 kW engine, 1200 cc, 1150 kg, 4.1 metres long
- Vehicle value: €8,500, vehicle age: 10 years, 4 doors
"""

HIGH_RISK_PROMPT = """
Please underwrite the following policy:

- Driver: 23 years old, has held a licence for 4 years
- Customer seniority: 1 year with this insurer
- Prior claims: 3 total, historical claim rate 0.75
- Policy is currently lapsed (is_active = 0)
- Policies in force: 1, max policies ever held: 1, max products: 1
- Second driver listed
- Area: 1, risk type: 3, distribution channel: 1, payment method: 1
- Vehicle: diesel, 180 kW engine, 2000 cc, 1650 kg, 4.5 metres long
- Vehicle value: €32,000, vehicle age: 2 years, 4 doors
"""

print("Scenarios defined.")
print("Starting agent runs — Ollama must be running locally.\n")

Scenarios defined.
Starting agent runs — Ollama must be running locally.



Defining two prompts for the LLM to process. We will also be testing some less efficient formatting.
Early prompt testing determined a risk of LLM hallucination on returning predicted values; LLM has been specifically forbidden returning values that are not returned by the tool.

### Step 6 — Run and Display

In [6]:
# ── Step 6: Run Agent and Display Memos ──────────────────────────────────────

display(Markdown("---"))
display(Markdown("## 📋 Agent Demo — Low-Risk Policy"))
display(Markdown("**Prompt sent to LLM:**"))
display(Markdown(f"```\n{LOW_RISK_PROMPT.strip()}\n```"))

memo_low = run_agent(LOW_RISK_PROMPT, verbose=True)
display(Markdown("**Underwriting Memo:**"))
display(Markdown(memo_low))

display(Markdown("---"))
display(Markdown("## 📋 Agent Demo — High-Risk Policy"))
display(Markdown("**Prompt sent to LLM:**"))
display(Markdown(f"```\n{HIGH_RISK_PROMPT.strip()}\n```"))

memo_high = run_agent(HIGH_RISK_PROMPT, verbose=True)
display(Markdown("**Underwriting Memo:**"))
display(Markdown(memo_high))

---

## 📋 Agent Demo — Low-Risk Policy

**Prompt sent to LLM:**

```
Please underwrite the following policy:

- Driver: 58 years old, has held a licence for 38 years
- Customer seniority: 7 years with this insurer
- Prior claims: 0 total, historical claim rate 0.0
- Policy is currently active
- Policies in force: 1, max policies ever held: 2, max products: 1
- No second driver listed
- Area: 0, risk type: 1, distribution channel: 0, payment method: 0
- Vehicle: petrol, 75 kW engine, 1200 cc, 1150 kg, 4.1 metres long
- Vehicle value: €8,500, vehicle age: 10 years, 4 doors
```


  → LLM called: predict_premium
    args: {'cylinder_capacity': 1200.0, 'vehicle_age': 10, 'second_driver': 0, 'payment': 0, 'seniority': 7, 'r_claims_history': 0.0, 'weight': 1150.0, 'length': 4.1, 'fuel_diesel': 0, 'area': 0, 'distribution_channel': 0, 'power': 75.0, 'value_vehicle': 8500.0, 'n_doors': 4, 'type_risk': 1, 'driver_age': 58, 'n_claims_history': 0, 'is_active': 1, 'policies_in_force': 1, 'max_policies': 2, 'max_products': 1, 'licence_years': 38}
    result: {'model_used': 'h1 — Random Forest', 'predicted_premium_euros': 265.87, 'risk_tier': 'Standard'}

  → LLM called: predict_claim_probability
    args: {'licence_years': 38, 'is_active': 1, 'second_driver': 0, 'cylinder_capacity': 1200.0, 'n_doors': 4, 'fuel_diesel': 0, 'r_claims_history': 0.0, 'max_products': 1, 'payment': 0, 'power': 75.0, 'weight': 1150.0, 'vehicle_age': 10, 'seniority': 7, 'n_claims_history': 0, 'policies_in_force': 1, 'max_policies': 2, 'distribution_channel': 0, 'value_vehicle': 8500.0, 'driver_a

...

```
Please underwrite the following policy:

- Driver: 23 years old, has held a licence for 4 years
- Customer seniority: 1 year with this insurer
- Prior claims: 3 total, historical claim rate 0.75
- Policy is currently lapsed (is_active = 0)
- Policies in force: 1, max policies ever held: 1, max products: 1
- Second driver listed
- Area: 1, risk type: 3, distribution channel: 1, payment method: 1
- Vehicle: diesel, 180 kW engine, 2000 cc, 1650 kg, 4.5 metres long
- Vehicle value: €32,000, vehicle age: 2 years, 4 doors
```


  → LLM called: predict_premium
    args: {'policies_in_force': 1, 'area': 1, 'distribution_channel': 1, 'fuel_diesel': 1, 'length': 4.5, 'n_claims_history': 3, 'r_claims_history': 0.75, 'second_driver': 1, 'cylinder_capacity': 2000.0, 'licence_years': 4, 'power': 180.0, 'seniority': 1, 'type_risk': 3, 'value_vehicle': 32000.0, 'driver_age': 23, 'max_policies': 1, 'payment': 1, 'is_active': 0, 'max_products': 1}
    result: {'model_used': 'h1 — Random Forest', 'predicted_premium_euros': 662.53, 'risk_tier': 'High Performance'}

  → LLM called: predict_claim_probability
    args: {'area': 1, 'cylinder_capacity': 2000.0, 'max_policies': 1, 'power': 180.0, 'r_claims_history': 0.75, 'value_vehicle': 32000.0, 'distribution_channel': 1, 'payment': 1, 'policies_in_force': 1, 'second_driver': 1, 'seniority': 1, 'driver_age': 23, 'fuel_diesel': 1, 'max_products': 1, 'n_claims_history': 3, 'is_active': 0, 'length': 4.5, 'licence_years': 4, 'type_risk': 3}
    result: {'model_used': 'h2 — Logist

c:\Users\Freddy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Freddy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


**Underwriting Memo:**

Underwriting Memo:

The predicted premium for this policy is €662.53. The predicted claim probability is 58.61%. Based on the risk tier classification, we are assigned a "High Performance" tier.

We chose to use the Random Forest model for predicting the premium due to its ability to handle complex interactions between features and its high accuracy in regression tasks.

The Logistic Regression model was chosen for predicting the claim probability because of its suitability for binary classification tasks and its historical performance in identifying drivers at risk of making a claim.

Given the predicted claim probability of 58.61%, we should be cautious when offering this policy to the customer. However, considering their seniority with our insurer (1 year) and relatively low claims history (3 total), it may still be worth proceeding with underwriting.

Success! The LLM combines logical explanations and interpretability that it is known for, while maintaining the accuracy and understanding of the trained ML models. To make absolutely sure, let's test a less "pretty" prompt.

### Step 8: Stress Test - Casual / Unstructured Input

In [7]:
# ── Step 8: Robustness Test — Casual / Unstructured Input ────────────────────
# The prompts above were well-formatted bullet lists. This tests whether the
# agent can still extract the necessary fields from the kind of casual,
# conversational input a real user might actually type.
# This is a meaningful test: the LLM's ability to parse unstructured language
# is precisely what distinguishes it from a traditional form-based input system.

CASUAL_PROMPT = """
hey can you check this out for me. got a guy, mid 30s maybe 34, been driving 
since he was like 18 so around 16 years. has 2 claims on his record, 
claim rate is about 0.4. hes been with us for 4 years. policy is still active.
he has 2 policies with us right now, max he ever had was 3, one product type.
no second driver. its a diesel, pretty average car — about 110 kw, 
1600cc engine, worth maybe 15000 euros, weighs around 1300 kg, 
roughly 4.3 metres. car is about 6 years old, 4 doors. area code 1, 
risk type 2, came through a broker (distribution channel 1), payment type 0.
"""

display(Markdown("---"))
display(Markdown("## 🧪 Robustness Test — Casual Unstructured Input"))
display(Markdown(
    "The following prompt mimics how a real user might casually describe a policy "
    "rather than filling out a structured form. This tests the LLM's ability to "
    "parse informal language and still route correctly to the right tools."
))
display(Markdown("**Prompt sent to LLM:**"))
display(Markdown(f"```\n{CASUAL_PROMPT.strip()}\n```"))

memo_casual = run_agent(CASUAL_PROMPT, verbose=True)

display(Markdown("**Underwriting Memo:**"))
display(Markdown(memo_casual))

---

## 🧪 Robustness Test — Casual Unstructured Input

The following prompt mimics how a real user might casually describe a policy rather than filling out a structured form. This tests the LLM's ability to parse informal language and still route correctly to the right tools.

**Prompt sent to LLM:**

```
hey can you check this out for me. got a guy, mid 30s maybe 34, been driving 
since he was like 18 so around 16 years. has 2 claims on his record, 
claim rate is about 0.4. hes been with us for 4 years. policy is still active.
he has 2 policies with us right now, max he ever had was 3, one product type.
no second driver. its a diesel, pretty average car — about 110 kw, 
1600cc engine, worth maybe 15000 euros, weighs around 1300 kg, 
roughly 4.3 metres. car is about 6 years old, 4 doors. area code 1, 
risk type 2, came through a broker (distribution channel 1), payment type 0.
```


  → LLM called: predict_premium
    args: {'driver_age': 34, 'is_active': 1, 'max_policies': 3, 'length': 4.3, 'r_claims_history': 0.4, 'policies_in_force': 2, 'max_products': 1, 'second_driver': 0, 'type_risk': 2, 'cylinder_capacity': 1600.0, 'licence_years': 16, 'area': 1, 'power': 110.0, 'weight': 1300.0, 'distribution_channel': 1, 'seniority': 4, 'n_claims_history': 2, 'fuel_diesel': 1, 'value_vehicle': 15000.0}
    result: {'model_used': 'h1 — Random Forest', 'predicted_premium_euros': 291.9, 'risk_tier': 'Premium'}

  → LLM called: predict_claim_probability
    args: {'weight': 1300.0, 'policies_in_force': 2, 'max_products': 1, 'fuel_diesel': 1, 'type_risk': 2, 'power': 110.0, 'cylinder_capacity': 1600.0, 'value_vehicle': 15000.0, 'driver_age': 34, 'licence_years': 16, 'seniority': 4, 'n_claims_history': 2, 'r_claims_history': 0.4, 'max_policies': 3, 'area': 1, 'distribution_channel': 1, 'is_active': 1, 'second_driver': 0, 'length': 4.3}
    result: {'model_used': 'h2 — Logistic

c:\Users\Freddy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
c:\Users\Freddy\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


**Underwriting Memo:**

Underwriting Memo:

The predicted premium for this policy is approximately €291.90, indicating a Premium risk tier. This prediction was made using our Random Forest regression model.

The predicted claim probability for this policy is 38.38%, indicating a moderate risk of claims. This prediction was made using our Logistic Regression classification model.

We chose the Random Forest model for premium prediction due to its strong performance in handling continuous variables, such as engine power and car value, which are relevant to determining premiums. The Logistic Regression model was chosen for claim probability prediction due to its effectiveness in handling binary outcomes and handling the number of claims history.

Based on these predictions, I recommend that we issue this policy with the Premium risk tier, but at a premium price of €291.90. This will ensure that we adequately compensate for the moderate level of claim risk associated with this driver's profile.

Still happy with that result. The LLM has successfully called the provided tools and processed the prompt. Potential extensions to this system include fail-safe checks for improper input formatting, utilizing an AI Agent system to refine the prompt before input into the tool-using LLM, or continuing to fine-tune the prompt itself.

## Create your own prompt below.

In [8]:
# ── Interactive Sandbox — enter any policy description below ─────────────
# Use natural language. The agent will parse it and call the appropriate tools.

custom_prompt = """

"""

if custom_prompt.strip():  # only run if the prompt is non-empty
    display(Markdown("---"))
    display(Markdown("## 🖊️ Custom Policy — Agent Output"))
    display(Markdown("**Prompt sent to LLM:**"))
    display(Markdown(f"```\n{custom_prompt.strip()}\n```"))
    memo_custom = run_agent(custom_prompt, verbose=True)
    display(Markdown("**Underwriting Memo:**"))
    display(Markdown(memo_custom))
else:
    print("Enter a policy description in custom_prompt above, then re-run this cell.")


Enter a policy description in custom_prompt above, then re-run this cell.


## Final Thoughts

While I am happy with this simplified LLM + Tool Library system, there are multiple extensions that can be utilized. 

The most significant extension would be RAG implementation, where the LLM is given access to a vector database and either pulls relevant example rows that are simpilar to the input row, OR the LLM pulls underwriting guidelines and applies that logic to each estimate. Both systems have their own pros and cons, but are likely more useful as AI companion tools for underwriters. A production extension would incorporate both RAG layers-- historical policy retrieval to ground predictions in comparable cases, and guideline retrieval to enforce regulatory compliance-- converting the system from a prediction narrator into a decision-support tool.

Companies like Verisk have already developed systems similar to what I described, and I have the opportunity to work with the team developing their MCP-based commercial underwriting tool this summer of 2026.